# RGT-Est horizon lab (experimental, GPU)

Experiments on how good the RGT horizons are, and how to make them better. Uses the same inputs as `RGT-Est_OwnData_Colab.ipynb`:

1. **Sparse picks:** are a few hand-picked lines enough? The guidance horizons are thinned to every Nth inline and crossline (as if picked by hand in RokDoc) and scored against the full horizons, **only on traces that weren't picked**.
2. **Flattened (Wheeler) view:** the seismic redrawn with RGT as the vertical axis. If the RGT is right, every reflection is a flat line. A **continuity** score (0–1) per level shows where it isn't.
3. **Snapping:** horizons traced from the RGT are moved onto the nearest peak/trough (±window), protected against jumps and crossings.
4. **Confidence:** the seismic's own dip vs the RGT-contour dip (ms per trace), plus reflector strength, turned into a 0–1 confidence volume.
5. **Repair:** between guidance horizons, where confidence is low, the RGT is blended towards proportional layering, then scored again with continuity.

**Before you start:** **Runtime → Change runtime type → L4 GPU**, then **Runtime → Run all**. Inputs come from Google Drive (`DRIVE_FOLDER`), and outputs go to `<DRIVE_FOLDER>/outputs/<date_time>_lab/`. RGT runs are cached in Drive, so later runs skip the model.

## 1. Settings

In [ ]:
DATA_SOURCE   = "drive"                           # "drive" or "upload"
DRIVE_FOLDER  = "/content/drive/MyDrive/PUBLIC"
DATA_FOLDER   = "/content"

MODEL_FILE    = "RGT-Est_CIG-Benchmark.pt"
SEGY_FILE     = "Full_Stack_8_41_CROP_1793889427461.sgy"
IL_BYTE, XL_BYTE = None, None
HORIZON_FILES = ["Jubilee.txt", "Ranger.txt"]

INTERVAL_MS   = (1300, 2300)
WELL_IL, WELL_XL = 1123, 2098                     # sections and horizon tracing go through here

SPARSE_PICK_EVERY = 10                            # sparse-picks test: keep every Nth inline and crossline (None = skip)
HORIZON_TIMES_AT_WELL = [1300, 1550, 1800, 2050, 2300]

SNAP_EVENT     = "auto"                           # "auto", "peak", "trough" or "zero"
SNAP_WINDOW_MS = "auto"                           # search ± this far (ms); "auto" = 30% of the dominant period
SNAP_MAX_JUMP_MS = 4                              # shifts this different from their neighbours fall back to the local median

DIP_TOL_MS_PER_TRACE = 0.25                       # dip misfit at which confidence drops to ~37%
REPAIR_STRENGTH = 1.0                             # 0 = no repair, 1 = full proportional layering where confidence is 0
CACHE_RGT      = True

## 2. Set up: GPU, Drive, code

In [ ]:
!pip -q install segyio
import os, glob, time, shutil, re, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
RUN_START = time.time()
print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("No GPU found. Runtime → Change runtime type → GPU.")

if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_FOLDER):
        raise FileNotFoundError(f"Drive folder not found: {DRIVE_FOLDER}")
    for f in [MODEL_FILE, SEGY_FILE] + list(HORIZON_FILES):
        src, dst = os.path.join(DRIVE_FOLDER, f), os.path.join(DATA_FOLDER, f)
        if not os.path.exists(src):
            print(f"not in Drive   {f}")
        elif os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            print(f"already here   {f}")
        else:
            t0 = time.time(); shutil.copyfile(src, dst)
            print(f"copied         {f}  ({os.path.getsize(dst)/1e6:.0f} MB, {time.time()-t0:.0f} s)")

def need(name):
    p = os.path.join(DATA_FOLDER, name)
    if not os.path.exists(p):
        raise FileNotFoundError(f"{name} not found. Files here: {sorted(os.listdir(DATA_FOLDER))}")
    return p
MODEL_PATH, SEGY_PATH = need(MODEL_FILE), need(SEGY_FILE)
HORIZON_PATHS = [need(h) for h in HORIZON_FILES]

!git clone -q https://github.com/fergusrssmith/RGT /content/RGT 2>/dev/null || git -C /content/RGT pull -q
sys.path[:0] = ["/content/RGT/colab", "/content/RGT/demo"]
import importlib, rgt_lfm as R, rgt_lab as L
importlib.reload(R); importlib.reload(L)

OUT_DIR = os.path.join(DRIVE_FOLDER if DATA_SOURCE == "drive" else DATA_FOLDER, "outputs",
                       time.strftime("%Y-%m-%d_%H%M", time.gmtime(RUN_START)) + "_lab")
os.makedirs(OUT_DIR, exist_ok=True)
CACHE_DIR = os.path.join(DRIVE_FOLDER if DATA_SOURCE == "drive" else DATA_FOLDER, "outputs", "_rgt_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
print("outputs →", OUT_DIR)

## 3. Seismic and guidance horizons

In [ ]:
G = R.read_segy_cube(SEGY_PATH, IL_BYTE, XL_BYTE)
cube, ilines, xlines, samples, dt = G["cube"], G["ilines"], G["xlines"], G["samples"], G["dt"]
NI, NX, NT = cube.shape
print(f"inlines {ilines[0]}–{ilines[-1]}, crosslines {xlines[0]}–{xlines[-1]}, TWT {samples[0]:.0f}–{samples[-1]:.0f} ms every {dt:g} ms")
ci = int(np.searchsorted(ilines, WELL_IL)) if WELL_IL in ilines else NI // 2
cx = int(np.searchsorted(xlines, WELL_XL)) if WELL_XL in xlines else NX // 2

HZ = [L.read_rokdoc_horizon(p, ilines, xlines, samples) for p in HORIZON_PATHS]
HZ = [h for h in HZ if np.isfinite(h["t"]).any()]
HZ.sort(key=lambda h: np.nanmean(h["t"]))
for h in HZ:
    print(f"{h['name']}: {np.isfinite(h['t']).mean()*100:.0f}% of traces, TWT {np.nanmin(h['t']):.0f}–{np.nanmax(h['t']):.0f} ms"
          + (f"   WARNING: Z type {h['ztype']}" if h["ztype"] and h["ztype"].upper() != "TWT" else ""))
HZ_SPARSE = [dict(name=h["name"], t=L.sparsify(h["t"], SPARSE_PICK_EVERY)) for h in HZ] if SPARSE_PICK_EVERY else []
if HZ_SPARSE:
    picked = np.isfinite(HZ_SPARSE[0]["t"])
    print(f"sparse picks: every {SPARSE_PICK_EVERY}th inline and crossline = {100*picked.mean():.0f}% of traces")

vclip = np.percentile(np.abs(cube), 99)
def seis_ax(ax, sec, ext, title):
    ax.imshow(sec, cmap="seismic_r", aspect="auto", vmin=-vclip, vmax=vclip, extent=ext)
    ax.set_title(title); ax.set_ylabel("TWT (ms)"); ax.set_ylim(ext[2], ext[3])
EXT_IL = (xlines[0], xlines[-1], samples[-1], samples[0])
fig, ax = plt.subplots(figsize=(16, 6))
seis_ax(ax, cube[ci].T, EXT_IL, f"inline {ilines[ci]} with guidance horizons")
for h in HZ:
    ax.plot(xlines, h["t"][ci], lw=2, label=h["name"])
ax.set_xlabel("crossline"); ax.legend(); plt.show()

## 4. Run RGT-Est (cached)

Three versions: **unguided**, **guided by the full horizons**, and **guided by sparse picks**. Each is saved in Drive after its first run.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from scipy.ndimage import grey_dilation
device = "cuda" if torch.cuda.is_available() else "cpu"
GRID = (400, 512, 512)
def z_score_clip(d, c=2.0):
    z = np.clip((d - d.mean()) / (d.std() + 1e-8), -c, c); return (z - z.min()) / (z.max() - z.min() + 1e-6)
seis_norm = z_score_clip(cube) * 2 - 1

def guidance_volume(use):
    ids = np.zeros((NI, NX, NT), np.int16)
    for k, h in enumerate(use, 1):
        i, j = np.nonzero(np.isfinite(h["t"]))
        ids[i, j, np.clip(np.round((h["t"][i, j] - samples[0]) / dt).astype(int), 0, NT - 1)] = k
    ids = grey_dilation(ids, size=2)
    vol = np.zeros((NI, NX, NT), np.float32)
    for k, h in enumerate(use, 1):
        vol[ids == k] = (np.nanmean(h["t"]) - samples[0]) / dt / NT * 2 - 1
    return vol

_model = {}
def run_rgt(use):
    if "m" not in _model:
        _model["m"] = torch.jit.load(MODEL_PATH, map_location=device).eval()
    s = F.interpolate(torch.from_numpy(seis_norm.transpose(2, 0, 1).copy())[None, None], GRID, mode="trilinear")
    g = torch.zeros_like(s) if not use else \
        F.interpolate(torch.from_numpy(guidance_volume(use).transpose(2, 0, 1).copy())[None, None], GRID, mode="nearest")
    x = torch.cat([s, g, (g != 0).float()], dim=1).to(device)
    t0 = time.time()
    with torch.no_grad(), torch.autocast(device_type=device):
        r = _model["m"](nn.ReplicationPad3d(8)(x))[:, :, 8:-8, 8:-8, 8:-8]
    out = F.interpolate(r.float(), (NT, NI, NX), mode="trilinear").cpu().numpy()[0, 0]
    del x, r
    if device == "cuda": torch.cuda.empty_cache()
    print(f"   inference {time.time()-t0:.1f} s")
    return z_score_clip(out).transpose(1, 2, 0).astype(np.float32)

st = os.stat(SEGY_PATH)
SEIS_KEY = f"{os.path.splitext(SEGY_FILE)[0]}_{st.st_size}_{int(st.st_mtime)}"
def rgt_cached(label, use, tag):
    key = SEIS_KEY + "_" + ("unguided" if not use else
                            tag + "_" + "+".join(f"{h['name']}-{int(np.nansum(h['t'])*10) % 10**8}" for h in use))
    path = os.path.join(CACHE_DIR, re.sub(r"[^A-Za-z0-9_.+-]", "_", key) + ".npy")
    if CACHE_RGT and os.path.exists(path):
        print(f"{label}: from cache"); return np.load(path)
    print(f"{label}: running the model"); out = run_rgt(use)
    if CACHE_RGT: np.save(path, out)
    return out

RGT = {"unguided": rgt_cached("unguided", [], "u")}
if HZ:
    RGT["guided"] = rgt_cached("guided (full horizons)", HZ, "full")
if HZ_SPARSE:
    RGT[f"guided, sparse 1/{SPARSE_PICK_EVERY}"] = rgt_cached(f"guided (sparse picks)", HZ_SPARSE, f"sparse{SPARSE_PICK_EVERY}")
BEST = "guided" if "guided" in RGT else "unguided"
print("versions:", list(RGT))

## 5. Test: how well does each version match the interpreted horizons?

Vertical miss (ms) between each interpreted horizon and the best-matching RGT surface, plus the hit rate within 2 samples.
For the **sparse** version, the miss is measured **only on traces that were not picked**, so it's a fair test of filling in between picked lines.
For **guided (full)**, the model saw every trace, so it only shows how closely the guidance is honoured.

In [ ]:
if not HZ:
    print("No horizons: skipped.")
else:
    not_picked = ~np.isfinite(HZ_SPARSE[0]["t"]) if HZ_SPARSE else None
    rows = []
    for name, vol in RGT.items():
        mask = not_picked if name.startswith("guided, sparse") else None
        for h, (e, hit) in zip(HZ, L.horizon_errors(vol, HZ, samples, mask=mask)):
            rows.append(dict(version=name, horizon=h["name"], miss_ms=e, hit_pct=hit,
                             measured_on="un-picked traces" if mask is not None else "all traces"))
    print(pd.DataFrame(rows).round(1).to_string(index=False))

## 6. Flattened (Wheeler) view and continuity

Each column shows the seismic along one inline, redrawn with **RGT as the vertical axis**. If the RGT is right, reflections are **flat and continuous**; bends and breaks show where it's wrong.
The continuity curve (right) averages several inlines: **1 = flat**, lower = the RGT drifts across reflections at that level (plotted at its typical TWT).

In [ ]:
levels = np.linspace(0.01, 0.99, 400)
check_inlines = np.linspace(0, NI - 1, 8, dtype=int)[1:-1]
CONT = {}
for name, vol in RGT.items():
    cs, ts = [], []
    for i in check_inlines:
        flat, t_at = L.flatten_section(cube[i], vol[i], samples, levels)
        cs.append(L.continuity(flat)); ts.append(np.nanmedian(t_at, axis=1))
    CONT[name] = (np.nanmean(cs, 0), np.nanmean(ts, 0))

n = len(RGT)
fig, axes = plt.subplots(1, n + 1, figsize=(6 * n + 5, 7), gridspec_kw={"width_ratios": [3] * n + [2]})
for ax, (name, vol) in zip(axes[:-1], RGT.items()):
    flat, t_at = L.flatten_section(cube[ci], vol[ci], samples, levels)
    ax.imshow(flat, cmap="seismic_r", aspect="auto", vmin=-vclip, vmax=vclip,
              extent=(xlines[0], xlines[-1], levels[-1], levels[0]))
    ax.set_title(f"{name}: inline {ilines[ci]} flattened"); ax.set_xlabel("crossline"); ax.set_ylabel("RGT")
for name, (c, t) in CONT.items():
    axes[-1].plot(c, t, lw=2, label=name)
axes[-1].set_ylim(samples[-1], samples[0]); axes[-1].set_xlim(0, 1); axes[-1].set_xlabel("continuity (1 = flat)")
axes[-1].set_ylabel("typical TWT (ms)"); axes[-1].legend(); axes[-1].set_title("continuity by level")
for t_ in INTERVAL_MS: axes[-1].axhline(t_, color="orange", ls="--", lw=1)
fig.tight_layout(); plt.show()
inside = lambda c, t: np.nanmean(c[(t >= INTERVAL_MS[0]) & (t <= INTERVAL_MS[1])])
print("mean continuity in the interval of interest:", {k: round(float(inside(*v)), 3) for k, v in CONT.items()})

## 7. Horizons from the RGT, snapped onto events

Horizons are traced through `HORIZON_TIMES_AT_WELL` at the chosen location using the best RGT version, then snapped to the nearest `SNAP_EVENT` within ±`SNAP_WINDOW_MS` (by default 30% of the dominant period, measured from the seismic).
Shifts that disagree with their neighbours by more than `SNAP_MAX_JUMP_MS` fall back to the local median, and horizons are never allowed to cross.

In [ ]:
f_dom = L.dominant_frequency(cube, samples, INTERVAL_MS)
win_ms = float(np.clip(0.3 * 1000 / f_dom, 4, 24)) if SNAP_WINDOW_MS == "auto" else float(SNAP_WINDOW_MS)
print(f"dominant frequency in the interval: {f_dom:.0f} Hz (period {1000/f_dom:.0f} ms) → snap window ±{win_ms:.0f} ms")
vol = RGT[BEST]
prof = L.monotonic(vol[ci, cx])
lv = np.interp(HORIZON_TIMES_AT_WELL, samples, prof)
HZ_RGT = L.trace_surfaces(vol, lv, samples)                             # (n, NI, NX) ms
HZ_SNAP, snap_stats, n_cross = L.snap_horizons(HZ_RGT, cube, samples, event=SNAP_EVENT,
                                              window_ms=win_ms, max_jump_ms=SNAP_MAX_JUMP_MS)
print(f"using the {BEST} RGT")
print(pd.DataFrame([dict(horizon=f"{t} ms", **s) for t, s in zip(HORIZON_TIMES_AT_WELL, snap_stats)]).round(2).to_string(index=False))
print(f"traces reverted to keep horizons from crossing: {n_cross}")

fig, axes = plt.subplots(2, 1, figsize=(16, 11))
for ax, (i_sec, lab) in zip(axes, ((ci, f"inline {ilines[ci]}"), (None, f"crossline {xlines[cx]}"))):
    if i_sec is not None:
        seis_ax(ax, cube[ci].T, EXT_IL, lab); xs = xlines; get = lambda a: a[ci]; ax.set_xlabel("crossline")
    else:
        seis_ax(ax, cube[:, cx].T, (ilines[0], ilines[-1], samples[-1], samples[0]), lab); xs = ilines
        get = lambda a: a[:, cx]; ax.set_xlabel("inline")
    for k in range(len(HZ_RGT)):
        ax.plot(xs, get(HZ_RGT[k]), color="k", lw=1, ls="--")
        ax.plot(xs, get(HZ_SNAP[k]), color="lime", lw=1.6)
    ax.set_title(lab + ": dashed = RGT horizon, green = snapped")
fig.tight_layout(); plt.show()

II, XX = np.meshgrid(ilines, xlines, indexing="ij")
stem = os.path.splitext(os.path.basename(SEGY_PATH))[0]
for tag, arr in (("RGT", HZ_RGT), ("RGT_snapped", HZ_SNAP)):
    for t_well, hz in zip(HORIZON_TIMES_AT_WELL, arr):
        pd.DataFrame({"Inline": II.ravel(), "Crossline": XX.ravel(), "X": G["X"].ravel(), "Y": G["Y"].ravel(),
                      "TWT_ms": hz.ravel()}).to_csv(os.path.join(OUT_DIR, f"{stem}_{tag}_{t_well}ms.txt"),
                                                    sep="\t", index=False, float_format="%.2f")
print("horizon files written to", OUT_DIR)

## 8. Confidence: does the RGT follow the reflections?

- **Dip misfit:** the difference between the seismic's local dip and the RGT contours' dip, in ms per trace.
- **Reflector strength (linearity):** how clearly layered the seismic is. Low means chaotic, faulted or noisy.
- **Confidence** = `exp(-(misfit / DIP_TOL)^2) × linearity`, from 0 to 1.

In [ ]:
t0 = time.time()
CONF = {k: L.confidence_volume(cube, RGT[k], dt, tol_ms_per_trace=DIP_TOL_MS_PER_TRACE)
        for k in ("unguided", "guided") if k in RGT}
print(f"computed in {time.time()-t0:.0f} s")
win = (samples >= INTERVAL_MS[0]) & (samples <= INTERVAL_MS[1])
for k, C in CONF.items():
    print(f"{k:9s} in the interval: median dip misfit {np.median(C['misfit'][..., win]):.2f} ms/trace, "
          f"confidence > 0.5 on {100*(C['confidence'][..., win] > 0.5).mean():.0f}% of samples")

n = len(CONF)
fig, axes = plt.subplots(n + 1, 1, figsize=(16, 5.2 * (n + 1)))
im = axes[0].imshow(next(iter(CONF.values()))["linearity"][ci].T, cmap="gray", aspect="auto", vmin=0, vmax=1, extent=EXT_IL)
axes[0].set_title(f"inline {ilines[ci]}: reflector strength (linearity); dark = chaotic/faulted"); plt.colorbar(im, ax=axes[0])
for ax, (k, C) in zip(axes[1:], CONF.items()):
    ax.imshow(cube[ci].T, cmap="gray", aspect="auto", vmin=-vclip, vmax=vclip, extent=EXT_IL)
    im = ax.imshow(C["confidence"][ci].T, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1, alpha=0.55, extent=EXT_IL)
    lv_ = np.linspace(0.02, 0.98, 40)
    ax.contour(RGT[k][ci].T, levels=lv_, colors="k", linewidths=0.5, extent=(xlines[0], xlines[-1], samples[0], samples[-1]))
    ax.set_title(f"{k}: confidence (green = RGT follows clear reflections, red = doesn't), black = RGT contours")
    plt.colorbar(im, ax=ax)
for ax in axes:
    ax.set_ylabel("TWT (ms)")
    for t_ in INTERVAL_MS: ax.axhline(t_, color="orange", ls="--", lw=1)
fig.tight_layout(); plt.show()

# confidence along each traced horizon
C = CONF[BEST]["confidence"]
fig, axes = plt.subplots(1, len(HZ_RGT), figsize=(4.2 * len(HZ_RGT), 4))
for ax, hz, t_well in zip(np.atleast_1d(axes), HZ_RGT, HORIZON_TIMES_AT_WELL):
    k = np.clip(np.round((hz - samples[0]) / dt).astype(int), 0, NT - 1)
    m = np.take_along_axis(C, k[..., None], 2)[..., 0]
    im = ax.imshow(m, cmap="RdYlGn", vmin=0, vmax=1, origin="lower", aspect="auto",
                   extent=(xlines[0], xlines[-1], ilines[0], ilines[-1]))
    ax.set_title(f"{t_well} ms horizon: mean {m.mean():.2f}"); ax.set_xlabel("crossline"); ax.set_ylabel("inline")
plt.colorbar(im, ax=np.atleast_1d(axes)[-1], label="confidence"); plt.show()

## 9. Repair (experimental): proportional layering where confidence is low

Between each pair of guidance horizons, the RGT is blended towards **proportional layering** (each point placed in proportion between the two horizons), with weight `REPAIR_STRENGTH × (1 − confidence)`. Near the horizons it ramps to zero, so they stay exactly where they are.
It's judged with the same flattening/continuity test as section 6. A repair that doesn't raise continuity isn't helping.

In [ ]:
if len(HZ) < 2 or "guided" not in RGT:
    print("Needs 2+ guidance horizons and the guided RGT: skipped.")
else:
    rep, wgt = L.proportional_repair(RGT["guided"], HZ, samples, CONF["guided"]["confidence"], strength=REPAIR_STRENGTH)
    RGT["guided + repair"] = rep
    cs, ts = [], []
    for i in check_inlines:
        flat, t_at = L.flatten_section(cube[i], rep[i], samples, levels)
        cs.append(L.continuity(flat)); ts.append(np.nanmedian(t_at, axis=1))
    CONT["guided + repair"] = (np.nanmean(cs, 0), np.nanmean(ts, 0))
    zone = (samples >= np.nanmin(HZ[0]["t"])) & (samples <= np.nanmax(HZ[-1]["t"]))
    print(f"blend weight between the horizons: mean {wgt[..., zone].mean():.2f}, max {wgt.max():.2f}")
    a0, b0 = np.nanmedian(HZ[0]["t"]), np.nanmedian(HZ[-1]["t"])
    for k in ("unguided", "guided", "guided + repair"):
        c, t = CONT[k]; m = (t >= a0) & (t <= b0)
        print(f"   continuity between {HZ[0]['name']} and {HZ[-1]['name']}: {k:16s} {np.nanmean(c[m]):.3f}")

    fig, axes = plt.subplots(1, 3, figsize=(22, 7), gridspec_kw={"width_ratios": [3, 3, 2]})
    for ax, k in zip(axes[:2], ("guided", "guided + repair")):
        flat, _ = L.flatten_section(cube[ci], RGT[k][ci], samples, levels)
        ax.imshow(flat, cmap="seismic_r", aspect="auto", vmin=-vclip, vmax=vclip, extent=(xlines[0], xlines[-1], levels[-1], levels[0]))
        ax.set_title(f"{k}: inline {ilines[ci]} flattened"); ax.set_xlabel("crossline"); ax.set_ylabel("RGT")
    for k, (c, t) in CONT.items():
        axes[2].plot(c, t, lw=2, label=k)
    axes[2].set_ylim(samples[-1], samples[0]); axes[2].set_xlim(0, 1); axes[2].legend(); axes[2].set_title("continuity by level")
    for h in HZ: axes[2].axhline(np.nanmedian(h["t"]), color="magenta", ls=":", lw=1)
    fig.tight_layout(); plt.show()

## 10. Save RGT volumes (SEG-Y, headers copied from the input)

In [ ]:
import segyio
def write_like_input(volume, path):
    shutil.copyfile(SEGY_PATH, path)
    with segyio.open(path, "r+", ignore_geometry=True) as f:
        for k in range(G["n_tr"]):
            f.trace[k] = np.ascontiguousarray(volume[G["il_idx"][k], G["xl_idx"][k]], dtype=np.float32)
    print("wrote", os.path.basename(path), f"({os.path.getsize(path)/1e6:.0f} MB)")
for k, vol in RGT.items():
    write_like_input(vol, os.path.join(OUT_DIR, f"{stem}_RGT_{re.sub(r'[^A-Za-z0-9]+', '_', k).strip('_')}.sgy"))
if "guided" in CONF:
    write_like_input(CONF["guided"]["confidence"], os.path.join(OUT_DIR, f"{stem}_RGT_confidence.sgy"))
print("all outputs in", OUT_DIR)